# 🎙️ Gemini-Based ASR Evaluation & Enrichment Pipeline

Multi-sheet Excel enrichment pipeline using **Gemini 3.1 Pro** for:
- Transcription (original language)
- Translation (to English)
- Match Scoring (semantic + WER + ROUGE-L + hallucination)
- 2-line Analysis

Results are written **in-place** into `MCF_Languages.xlsx` — existing values are never overwritten.

## Cell 1 — Install Dependencies

In [35]:
!pip install google-genai python-dotenv openpyxl pandas sentence-transformers bert-score jiwer rouge-score tenacity -q

## Cell 2 — Configuration

In [36]:
import os

# ─── Paths ─────────────────────────────────────────────────────────────────────
AUDIO_ROOT  = "C:/Users/gayat/Desktop/transcription_script/audio_files"          # ./audio_files/<language>/<file>
EXCEL_PATH  = "C:/Users/gayat/Desktop/transcription_script/transcription_data/MCF_Languages.xlsx"   # Multi-sheet workbook

# ─── Model ─────────────────────────────────────────────────────────────────────
TRANSCRIPTION_MODEL = "gemini-3.1-pro-preview"
TRANSLATION_MODEL   = "gemini-3.1-pro-preview"
ANALYSIS_MODEL      = "gemini-3.1-pro-preview"

# ─── Supported audio formats ────────────────────────────────────────────────────
SUPPORTED_FORMATS = [".wav", ".mp3", ".m4a", ".flac"]

# ─── Resilience ────────────────────────────────────────────────────────────────
MAX_RETRIES   = 1
PROCESS_MODE  = "sequential"

# ─── Output column names ────────────────────────────────────────────────────────
COL_TRANSCRIPT   = "Gemini_3.1_Pro_Trancript"        # sic — matches design doc
COL_TRANSLATION  = "Gemini_3.1_Pro_Translation"
COL_MATCH_PCT    = "Gemini_3.1_Pro_Match_Percentage"
COL_WER          = "Gemini_3.1_Pro_WER"
COL_ANALYSIS     = "Gemini_3.1_Analysis"

OUTPUT_COLUMNS = [COL_TRANSCRIPT, COL_TRANSLATION, COL_MATCH_PCT, COL_WER, COL_ANALYSIS]

print("✅ Configuration loaded")
print(f"   AUDIO_ROOT  : {os.path.abspath(AUDIO_ROOT)}")
print(f"   EXCEL_PATH  : {os.path.abspath(EXCEL_PATH)}")
print(f"   MODEL       : {TRANSCRIPTION_MODEL}")

✅ Configuration loaded
   AUDIO_ROOT  : C:\Users\gayat\Desktop\transcription_script\audio_files
   EXCEL_PATH  : C:\Users\gayat\Desktop\transcription_script\transcription_data\MCF_Languages.xlsx
   MODEL       : gemini-3.1-pro-preview


## Cell 3 — Imports

In [37]:
import os
import time
import logging
import warnings
import re
from pathlib import Path
from typing import Optional

import pandas as pd
import numpy as np
from dotenv import load_dotenv

# Gemini
from google import genai
from google.genai import types

# Scoring libraries
from jiwer import wer as compute_wer                 # WER
from rouge_score import rouge_scorer                  # ROUGE-L
from sentence_transformers import SentenceTransformer, util  # Semantic similarity

warnings.filterwarnings("ignore")

# ─── Logging setup ─────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("asr_pipeline")

print("✅ Imports complete")

✅ Imports complete


## Cell 4 — Gemini Initialization

In [38]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

if not api_key or api_key.startswith("your_"):
    raise ValueError("❌ Please set GEMINI_API_KEY in your .env file")

client = genai.Client(api_key=api_key)
print("✅ Gemini client initialized")
print(f"   Key ends with: ...{api_key[-6:]}")

✅ Gemini client initialized
   Key ends with: ...htWJVY


## Cell 5 — Utility Functions

In [39]:
# ─────────────────────────────────────────────────────────────────────────────
# Load the sentence-transformer model once (all-MiniLM is fast & multilingual-ish)
# For true multilingual support you can swap to "sentence-transformers/LaBSE"
# ─────────────────────────────────────────────────────────────────────────────
print("🔄 Loading sentence-transformer model...")
_ST_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("✅ Sentence-transformer ready")

_ROUGE = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)


# ── 1. Semantic Similarity ────────────────────────────────────────────────────
def compute_semantic_similarity(ref: str, hyp: str) -> float:
    """Cosine similarity of sentence embeddings.  Returns 0–1."""
    if not ref.strip() or not hyp.strip():
        return 0.0
    embs = _ST_MODEL.encode([ref, hyp], convert_to_tensor=True)
    score = float(util.cos_sim(embs[0], embs[1]))
    return max(0.0, min(1.0, score))


# ── 2. Word Error Rate ────────────────────────────────────────────────────────
def compute_clipped_wer(reference: str, hypothesis: str) -> float:
    """
    Computes WER and clips it to [0, 1].
    Formula: WER = (S + D + I) / N,  clipped_WER = min(WER, 1.0)
    """
    if not reference.strip():
        return 0.0
    if not hypothesis.strip():
        return 1.0
    try:
        raw_wer = compute_wer(reference, hypothesis)
        return min(float(raw_wer), 1.0)
    except Exception:
        return 1.0


# ── 3. Sequential Similarity (ROUGE-L) ────────────────────────────────────────
def compute_rouge_l(reference: str, hypothesis: str) -> float:
    """ROUGE-L F1 score (Longest Common Subsequence).  Returns 0–1."""
    if not reference.strip() or not hypothesis.strip():
        return 0.0
    try:
        scores = _ROUGE.score(reference, hypothesis)
        return float(scores["rougeL"].fmeasure)
    except Exception:
        return 0.0


# ── 4. Find audio file in language folder ─────────────────────────────────────
def find_audio_file(language: str, filename: str) -> Optional[str]:
    """
    Constructs: AUDIO_ROOT/<language>/<filename>
    Returns absolute path if found, else None.
    Tries case-insensitive subfolder matching as a fallback.
    """
    base = Path(AUDIO_ROOT)
    # Direct match
    direct = base / language / filename
    if direct.exists():
        return str(direct)

    # Case-insensitive folder search
    if base.exists():
        for folder in base.iterdir():
            if folder.is_dir() and folder.name.lower() == language.lower():
                candidate = folder / filename
                if candidate.exists():
                    return str(candidate)
                # Try case-insensitive filename match
                for f in folder.iterdir():
                    if f.name.lower() == filename.lower():
                        return str(f)
    return None


# ── 5. Check if a cell already has a usable value ────────────────────────────
def is_filled(value) -> bool:
    """Returns True if the cell already contains a non-NaN, non-empty value."""
    if value is None:
        return False
    if isinstance(value, float) and np.isnan(value):
        return False
    if isinstance(value, str) and value.strip() == "":
        return False
    return True


print("✅ Utility functions defined")

13:03:04  INFO      Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


🔄 Loading sentence-transformer model...


13:03:07  INFO      Use pytorch device_name: cpu
13:03:07  INFO      Using default tokenizer.


✅ Sentence-transformer ready
✅ Utility functions defined


## Cell 6 — Agents

In [40]:
import time


# ─────────────────────────────────────────────────────────────────────────────
# Helper: call Gemini with retry-once logic
# ─────────────────────────────────────────────────────────────────────────────
def _gemini_call(model: str, contents: list, retries: int = MAX_RETRIES) -> str:
    """Calls Gemini and returns response.text. Retries once on failure."""
    for attempt in range(retries + 1):
        try:
            response = client.models.generate_content(
                model=model,
                contents=contents,
            )
            return response.text.strip()
        except Exception as exc:
            if attempt < retries:
                log.warning(f"Gemini call failed (attempt {attempt+1}): {exc}. Retrying...")
                time.sleep(3)
            else:
                log.error(f"Gemini call failed after {retries+1} attempt(s): {exc}")
                raise


# ─────────────────────────────────────────────────────────────────────────────
# Agent 1 — Transcription Agent
# ─────────────────────────────────────────────────────────────────────────────
def transcription_agent(audio_path: str, language: str) -> str:
    """
    Upload audio to Gemini Files API and transcribe in the original language.
    Returns the raw transcription string.
    """
    log.info(f"  📤 Uploading: {Path(audio_path).name}")
    uploaded = client.files.upload(file=audio_path)

    prompt = (
        f"Transcribe the audio exactly in its original language ({language}). "
        "Do not translate. Do not summarize. "
        "Output ONLY the plain transcription text with no timestamps, "
        "no speaker labels, no formatting, no headers."
    )

    log.info("  🧠 Transcribing...")
    t0 = time.time()
    result = _gemini_call(TRANSCRIPTION_MODEL, [uploaded, prompt])
    log.info(f"  ✅ Transcription done in {time.time()-t0:.1f}s")

    # Best-effort cleanup
    try:
        client.files.delete(name=uploaded.name)
    except Exception:
        pass

    return result


# ─────────────────────────────────────────────────────────────────────────────
# Agent 2 — Translation Agent
# ─────────────────────────────────────────────────────────────────────────────
def translation_agent(text: str) -> str:
    """
    Translates `text` to English using Gemini.
    Returns the English translation.
    """
    if not text or not text.strip():
        return ""

    prompt = (
        "Translate the following text to English. "
        "Preserve meaning exactly. Do not summarize. "
        "Output only the translation with no extra commentary.\n\n"
        f"{text}"
    )

    t0 = time.time()
    result = _gemini_call(TRANSLATION_MODEL, [prompt])
    log.info(f"  🌐 Translation done in {time.time()-t0:.1f}s")
    return result


# ─────────────────────────────────────────────────────────────────────────────
# Agent 3 — Scoring Agent
# ─────────────────────────────────────────────────────────────────────────────
def _hallucination_score(original: str, asr_output: str) -> float:
    """
    LLM-based hallucination scoring (0 = none, 1 = severe).
    Returns a float in [0, 1].
    """
    if not original.strip() or not asr_output.strip():
        return 1.0

    prompt = (
        "Compare the original transcript and the ASR output below.\n"
        "Score hallucination from 0 to 1:\n"
        "  0 = no hallucination\n"
        "  1 = severe hallucination\n"
        "Return ONLY a single decimal number, nothing else.\n\n"
        f"Original transcript:\n{original}\n\n"
        f"ASR output:\n{asr_output}"
    )
    try:
        raw = _gemini_call(ANALYSIS_MODEL, [prompt])
        # Extract first float-like token
        match = re.search(r"[0-9]*\.?[0-9]+", raw)
        if match:
            return max(0.0, min(1.0, float(match.group())))
    except Exception as exc:
        log.warning(f"  Hallucination scoring failed: {exc}")
    return 0.5  # neutral fallback


def scoring_agent(original: str, hypothesis: str) -> dict:
    """
    Computes the full match score and WER.

    Returns:
        {
            'match_percentage': float (0–100),
            'wer': float (0–1, clipped),
            'semantic': float,
            'rouge_l': float,
            'hallucination': float,
        }

    Final score formula:
        match_score = 0.35 * semantic
                    + 0.30 * (1 - clipped_WER)
                    + 0.20 * rouge_l
                    + 0.15 * (1 - hallucination)
    """
    if not original.strip():
        return {"match_percentage": 0.0, "wer": 1.0, "semantic": 0.0,
                "rouge_l": 0.0, "hallucination": 1.0}

    semantic      = compute_semantic_similarity(original, hypothesis)
    clipped_wer   = compute_clipped_wer(original, hypothesis)
    rouge_l       = compute_rouge_l(original, hypothesis)
    hallucination = _hallucination_score(original, hypothesis)

    match_score = (
        0.35 * semantic
        + 0.30 * (1 - clipped_wer)
        + 0.20 * rouge_l
        + 0.15 * (1 - hallucination)
    )
    match_percentage = round(match_score * 100, 2)

    log.info(
        f"  📊 Scores — Semantic: {semantic:.3f} | WER: {clipped_wer:.3f} | "
        f"ROUGE-L: {rouge_l:.3f} | Hallucination: {hallucination:.3f} | "
        f"Match: {match_percentage}%"
    )

    return {
        "match_percentage": match_percentage,
        "wer": round(clipped_wer, 4),
        "semantic": round(semantic, 4),
        "rouge_l": round(rouge_l, 4),
        "hallucination": round(hallucination, 4),
    }


# ─────────────────────────────────────────────────────────────────────────────
# Agent 4 — Analysis Agent
# ─────────────────────────────────────────────────────────────────────────────
def analysis_agent(
    original: str,
    asr_output: str,
    match_pct: float,
    wer: float,
) -> str:
    """
    Generates a concise 2-line quality analysis.
    """
    if not original.strip() or not asr_output.strip():
        return "No valid transcript available for analysis."

    prompt = (
        "Explain why the transcription quality is low or high.\n"
        "Focus on: missing words, incorrect meaning, hallucinations.\n"
        "Answer in EXACTLY 2 lines. No bullet points, no headers.\n\n"
        f"Original transcript:\n{original}\n\n"
        f"ASR output:\n{asr_output}\n\n"
        f"Match Score: {match_pct}%  |  WER: {wer}"
    )

    try:
        result = _gemini_call(ANALYSIS_MODEL, [prompt])
        # Ensure at most 2 lines are returned
        lines = [l.strip() for l in result.strip().splitlines() if l.strip()]
        return " ".join(lines[:2]) if lines else result
    except Exception as exc:
        log.warning(f"  Analysis agent failed: {exc}")
        return "Analysis unavailable due to API error."


print("✅ All 4 agents defined")

✅ All 4 agents defined


## Cell 7 — Core Row Processor

In [41]:
def process_row(
    row: pd.Series,
    language: str,
    row_idx: int,
) -> dict:
    """
    Processes a single Excel row end-to-end.

    Input row expected fields:
        Audio_Samples        – filename (e.g. sample1.wav, sample1.flac)
        Original_Transcript  – ground-truth text

    Returns a dict with keys matching OUTPUT_COLUMNS.
    """
    result = {
        COL_TRANSCRIPT:  None,
        COL_TRANSLATION: None,
        COL_MATCH_PCT:   None,
        COL_WER:         None,
        COL_ANALYSIS:    None,
    }

    # ── Pull fields safely ────────────────────────────────────────────────────
    audio_filename   = str(row.get("Audio_Samples", "")).strip()
    original_text    = str(row.get("Original_Transcript", "")).strip()

    # Skip rows without an audio filename
    if not audio_filename or audio_filename.lower() in ("nan", ""):
        log.warning(f"  Row {row_idx}: No Audio_Samples value — skipping")
        return result

    # ── Locate audio file ─────────────────────────────────────────────────────
    audio_path = find_audio_file(language, audio_filename)
    if audio_path is None:
        log.warning(f"  Row {row_idx}: Audio not found — {language}/{audio_filename}")
        result[COL_ANALYSIS] = f"Audio file '{audio_filename}' not found in {language} folder."
        return result

    # ── Already filled? Skip per-column ──────────────────────────────────────
    need_transcript   = not is_filled(row.get(COL_TRANSCRIPT))
    need_translation  = not is_filled(row.get(COL_TRANSLATION))
    need_scores       = (not is_filled(row.get(COL_MATCH_PCT))) or (not is_filled(row.get(COL_WER)))
    need_analysis     = not is_filled(row.get(COL_ANALYSIS))

    if not any([need_transcript, need_translation, need_scores, need_analysis]):
        log.info(f"  Row {row_idx}: All columns already filled — skipping")
        return result  # all None → caller will keep existing values

    # ── Step 1: Transcription ─────────────────────────────────────────────────
    transcript = row.get(COL_TRANSCRIPT) if is_filled(row.get(COL_TRANSCRIPT)) else None

    if need_transcript:
        try:
            transcript = transcription_agent(audio_path, language)
            result[COL_TRANSCRIPT] = transcript
        except Exception as exc:
            log.error(f"  Row {row_idx}: Transcription FAILED — {exc}")
            result[COL_ANALYSIS] = f"Transcription error: {exc}"
            return result
    else:
        transcript = str(row.get(COL_TRANSCRIPT, "")).strip()

    if not transcript:
        log.warning(f"  Row {row_idx}: Empty transcription returned")
        result[COL_ANALYSIS] = "Transcription returned empty output."
        return result

    # ── Step 2: Translation ───────────────────────────────────────────────────
    translation = row.get(COL_TRANSLATION) if is_filled(row.get(COL_TRANSLATION)) else None

    if need_translation:
        try:
            translation = translation_agent(transcript)
            result[COL_TRANSLATION] = translation
        except Exception as exc:
            log.error(f"  Row {row_idx}: Translation FAILED — {exc}")
            translation = ""

    # ── Step 3: Scoring ───────────────────────────────────────────────────────
    scores = None
    if need_scores and original_text:
        try:
            scores = scoring_agent(original_text, transcript)
            result[COL_MATCH_PCT] = scores["match_percentage"]
            result[COL_WER]       = scores["wer"]
        except Exception as exc:
            log.error(f"  Row {row_idx}: Scoring FAILED — {exc}")
    elif not original_text:
        log.warning(f"  Row {row_idx}: Original_Transcript empty — scoring skipped")

    # ── Step 4: Analysis ──────────────────────────────────────────────────────
    if need_analysis and original_text:
        match_pct = scores["match_percentage"] if scores else 0.0
        wer_val   = scores["wer"]              if scores else 1.0
        try:
            analysis = analysis_agent(original_text, transcript, match_pct, wer_val)
            result[COL_ANALYSIS] = analysis
        except Exception as exc:
            log.error(f"  Row {row_idx}: Analysis FAILED — {exc}")

    return result


print("✅ Row processor defined")

✅ Row processor defined


## Cell 8 — Excel Processing Pipeline

In [42]:
import pandas as pd


def ensure_output_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Adds any missing output columns as NaN columns at the end."""
    for col in OUTPUT_COLUMNS:
        if col not in df.columns:
            df[col] = None
    return df


def run_pipeline():
    """
    Main execution function.
    - Reads all sheets from EXCEL_PATH
    - Processes each row sequentially
    - Writes results back in-place after every sheet
    """
    excel_path = os.path.abspath(EXCEL_PATH)

    if not os.path.exists(excel_path):
        raise FileNotFoundError(f"❌ Excel file not found: {excel_path}")

    print(f"\n📂 Reading workbook: {excel_path}")
    all_sheets: dict[str, pd.DataFrame] = pd.read_excel(
        excel_path, sheet_name=None, engine="openpyxl"
    )
    sheet_names = list(all_sheets.keys())
    print(f"   Found {len(sheet_names)} sheet(s): {sheet_names}\n")

    # Log trackers
    missing_files: list[str]  = []
    api_failures:  list[str]  = []
    low_scores:    list[str]  = []

    for sheet_name in sheet_names:
        df = all_sheets[sheet_name]
        language = sheet_name.strip()

        # ── Validate required columns ─────────────────────────────────────────
        if "Audio_Samples" not in df.columns:
            log.warning(f"Sheet '{sheet_name}': missing 'Audio_Samples' column — skipping")
            continue

        print(f"{'─'*60}")
        print(f"🗂️  Sheet: {sheet_name}  |  Language: {language}  |  Rows: {len(df)}")
        print(f"{'─'*60}")

        # Ensure output columns exist
        df = ensure_output_columns(df)

        for row_idx, row in df.iterrows():
            print(f"\n  → Row {row_idx + 1}/{len(df)}  |  {row.get('Audio_Samples', '?')}")

            try:
                updates = process_row(row, language, row_idx)
            except Exception as exc:
                log.error(f"  Row {row_idx}: Unexpected error — {exc}")
                api_failures.append(f"{sheet_name} / row {row_idx}: {exc}")
                continue

            # ── Merge updates back into DataFrame (skip None = keep original) ─
            for col, value in updates.items():
                if value is not None:
                    df.at[row_idx, col] = value

            # ── Log low match scores ──────────────────────────────────────────
            match_pct = df.at[row_idx, COL_MATCH_PCT]
            if is_filled(match_pct) and float(match_pct) < 50:
                low_scores.append(
                    f"{sheet_name} / {row.get('Audio_Samples')} — {match_pct}%"
                )

        # ── Save after each sheet (safe checkpoint) ───────────────────────────
        all_sheets[sheet_name] = df

        print(f"\n  💾 Saving checkpoint after sheet '{sheet_name}'...")
        with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
            for sn, sdf in all_sheets.items():
                sdf.to_excel(writer, sheet_name=sn, index=False)
        print(f"  ✅ Sheet '{sheet_name}' saved")

    # ── Final Summary ─────────────────────────────────────────────────────────
    print(f"\n{'═'*60}")
    print("🏁  PIPELINE COMPLETE")
    print(f"{'═'*60}")
    print(f"  Sheets processed : {len(sheet_names)}")
    print(f"  Missing files    : {len(missing_files)}")
    print(f"  API failures     : {len(api_failures)}")
    print(f"  Low scores (<50%): {len(low_scores)}")

    if missing_files:
        print("\n⚠️  Missing audio files:")
        for m in missing_files:
            print(f"    {m}")

    if api_failures:
        print("\n❌  API failures:")
        for f in api_failures:
            print(f"    {f}")

    if low_scores:
        print("\n🔴  Low match scores (<50%):")
        for s in low_scores:
            print(f"    {s}")

    print(f"\n📄 Output written to: {excel_path}")


print("✅ Pipeline function defined — run Cell 9 to execute")

✅ Pipeline function defined — run Cell 9 to execute


## Cell 9 — ▶️ Execute Pipeline

> **Before running:** Make sure
> - `MCF_Languages.xlsx` is in the same folder as this notebook
> - Audio files are at `./audio_files/<SheetName>/<filename>`
> - Your `.env` contains `GEMINI_API_KEY=...`

In [43]:
run_pipeline()

13:03:24  WARNING     Row 0: No Audio_Samples value — skipping
13:03:24  INFO        📤 Uploading: amharic_sample_1.flac



📂 Reading workbook: C:\Users\gayat\Desktop\transcription_script\transcription_data\MCF_Languages.xlsx
   Found 1 sheet(s): ['Amharic']

────────────────────────────────────────────────────────────
🗂️  Sheet: Amharic  |  Language: Amharic  |  Rows: 4
────────────────────────────────────────────────────────────

  → Row 1/4  |  nan

  → Row 2/4  |  amharic_sample_1.flac


13:03:25  INFO      HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files "HTTP/1.1 200 OK"
13:03:28  INFO      HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files?upload_id=AMNfjG0xQFs49y7XvIom3_PrrcrsBMgpNf8L6QvvMPFRX37ztR8a61K09prRwclmjeLWXABMB_WU7vTDy1XfZjWc_3al5PFAou8x-lGBuiplRW0&upload_protocol=resumable "HTTP/1.1 200 OK"
13:03:28  INFO        🧠 Transcribing...
13:03:28  INFO      AFC is enabled with max remote calls: 10.
13:03:49  INFO      HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"
13:03:49  INFO        ✅ Transcription done in 21.3s
13:03:51  INFO      HTTP Request: DELETE https://generativelanguage.googleapis.com/v1beta/files/4ooqna3gkulw "HTTP/1.1 200 OK"
13:03:51  INFO      AFC is enabled with max remote calls: 10.
13:04:26  INFO      HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-pro-preview:g


  → Row 3/4  |  amharic_sample_2.flac


13:04:26  INFO      HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files "HTTP/1.1 200 OK"
13:04:28  INFO      HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files?upload_id=AMNfjG3IYpJ1oYTbBjSdSuTKNgyJyjGIJtj98PBnxpd42nzatx7A3JrwJNUJO1y2QsZtTIPPYiDMuS7y7StH2KkaClWkzXHZQx8NnVdBxsHaBw&upload_protocol=resumable "HTTP/1.1 200 OK"
13:04:28  INFO        🧠 Transcribing...
13:04:28  INFO      AFC is enabled with max remote calls: 10.
13:04:59  INFO      HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"
13:04:59  INFO        ✅ Transcription done in 30.8s
13:05:01  INFO      HTTP Request: DELETE https://generativelanguage.googleapis.com/v1beta/files/14nhly7x41wc "HTTP/1.1 200 OK"
13:05:01  INFO      AFC is enabled with max remote calls: 10.
13:05:15  INFO      HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-pro-preview:ge


  → Row 4/4  |  amharic_sample_3.flac


13:05:16  INFO      HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files "HTTP/1.1 200 OK"
13:05:18  INFO      HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files?upload_id=AMNfjG0NGV0hJMKwzdgM452tW_l6kRkxqZRRyOUFtBKUHeoPWhzJbcH_WJr_W2jLTnbRpzfhTu5E71lqloOdM_JU_rvx5ASDfOgXZ9Yh6Y1PBm0&upload_protocol=resumable "HTTP/1.1 200 OK"
13:05:18  INFO        🧠 Transcribing...
13:05:18  INFO      AFC is enabled with max remote calls: 10.
13:06:00  INFO      HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"
13:06:00  INFO        ✅ Transcription done in 42.1s
13:06:02  INFO      HTTP Request: DELETE https://generativelanguage.googleapis.com/v1beta/files/6utbu5qb9y7g "HTTP/1.1 200 OK"
13:06:02  INFO      AFC is enabled with max remote calls: 10.
13:09:57  INFO      HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-pro-preview:g


  💾 Saving checkpoint after sheet 'Amharic'...
  ✅ Sheet 'Amharic' saved

════════════════════════════════════════════════════════════
🏁  PIPELINE COMPLETE
════════════════════════════════════════════════════════════
  Sheets processed : 1
  Missing files    : 0
  API failures     : 0
  Low scores (<50%): 0

📄 Output written to: C:\Users\gayat\Desktop\transcription_script\transcription_data\MCF_Languages.xlsx


---
## 📋 Optional — Quick Results Preview

In [14]:
# After the pipeline runs, use this cell to preview any sheet
import pandas as pd

PREVIEW_SHEET = None  # Set to a sheet name, e.g. "Hausa", or None to show all

all_sheets = pd.read_excel(EXCEL_PATH, sheet_name=None, engine="openpyxl")
sheets_to_show = [PREVIEW_SHEET] if PREVIEW_SHEET else list(all_sheets.keys())

for sheet in sheets_to_show:
    if sheet not in all_sheets:
        print(f"Sheet '{sheet}' not found")
        continue
    df = all_sheets[sheet]
    preview_cols = [c for c in ["Audio_Samples", "Original_Transcript"] + OUTPUT_COLUMNS if c in df.columns]
    print(f"\n{'═'*70}")
    print(f"  Sheet: {sheet}  ({len(df)} rows)")
    print(f"{'═'*70}")
    display(df[preview_cols].head(10))


══════════════════════════════════════════════════════════════════════
  Sheet: Sheet1  (0 rows)
══════════════════════════════════════════════════════════════════════


,Audio_Samples,Gemini_3.1_Pro_Trancript,Gemini_3.1_Pro_Translation,Gemini_3.1_Pro_Match_Percentage,Gemini_3.1_Pro_WER,Gemini_3.1_Analysis
